# Notebook for plotting Cuk+ (2021) evolution Movies S2-S3 to accompany Figures 7,8, and S1
## Preamble
### Load packages required

In [1]:
#SJL 1/2020
#Script to plot the change in surface length for planets for an example tidal evolution
#plots on a 2D map showing the change in shape and surface


###########################################################
###########################################################
###########################################################
import numpy as np
import scipy as sp
import sys
import os
import struct
from scipy import constants as const

from scipy.signal import savgol_filter

#package to use wildcards 
import fnmatch

import csv

from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import griddata

from scipy import interpolate

#plotting packages
import matplotlib as mpl
import matplotlib.pyplot as plt
import pylab
import matplotlib.cm as cm
from matplotlib import gridspec


cwd = os.getcwd()
print(cwd)
if sys.platform== 'darwin':
    sys.path.insert(0, cwd+'/Support_scipts')
    print(cwd+'/Support_scripts')
elif (sys.platform== 'win32') | (sys.platform== 'win64'):
    sys.path.insert(0,cwd+"\\Support_scipts")
    
#HERCULES_structures
from HERCULES_structures import *
from surface_size_calc import *
from HERCULES_random_planet_database_structure_1D import *

#functions for calculating non-evenly spaced numerical differentials
from gradients import *

#import colormaps
import colormaps as cmaps
import matplotlib.cm as cm

import svglib.svglib as svglib
svglib.register_font('helvetica', './Helvetica.ttc')

/Users/vq21447/Documents/Lock_2026_SI
/Users/vq21447/Documents/Lock_2026_SI/Support_scripts
CHECK THE LOCATION OF ODYSSEY BACKUP
CHECK THE LOCATION OF ODYSSEY BACKUP


('helvetica', True)

### Define constants

In [2]:
########################################################################################
########################################################################################
########################################################################################
#CONSTANTS
MEarth=5.972E24
LEM=3.5E34
REarth=6.371E6
MMoon=7.34767309E22

aMoon=0.3844E9
aCassini=30*REarth
aRoche=2.9*REarth

#for HERCULES
MEarth_H=5.9879648E24
LEM_H=3.53E34

### Set parameters and scenario to plot

In [3]:
########################################################################################
########################################################################################
########################################################################################
#PARAMS

#info for HERCULES arrays
Hdir='Earth_correct_params_S3.20c'
Hname='Earth_correct_params_S3.20c'
    

#directory for max change of length data
data_dir='Data/Cuk_et_al_2021_high_obliquity'

#which plot do you want
#0: Figure 3 of Cuk et al 2021 (Movie S3)
#1: Figure 7 of Cuk et al 2021 (Movie S2)
flag_data=1

#number of contor points
Ncont=1000 #500

#time step for plotting
tstep_plot=0.1 #Myr

#Earth's moment of inertia used to convert to AM
C_Earth=0.3304

#output for snapshots
if flag_data==0:
    output_dir='Movie_slides/MovieS3_slides'
elif flag_data==1:
    output_dir='Movie_slides/MovieS2_slides'

if os.path.isdir(output_dir)==False:
    os.mkdir(output_dir)

## Main script
### Read in HERCULES database

In [4]:
########################################################################################
########################################################################################
########################################################################################
#MAIN

########################################################################################
#read in the database

Hdatabase=HERCULES_random_planet_database_1D()
Hdatabase.make_array(Hdir,Hname)
Hdatabase.initialize_interpolation([0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1],\
                                   [0,0,[0],0,0,0,0,[0],[0],[0],[0],[0],[0],[0],[0],[0],[0],[0]], flag_extrap=1)

#extract the latitudes for each Mu point
Nmu=Hdatabase.parr[0].Nmu
lat=np.arccos(Hdatabase.parr[0].layers[0].mu)*180/np.pi

Earth_correct_params_S3.20c
	 Earth_correct_params_S3.20c_AM3.3125
	 Earth_correct_params_S3.20c_AM3.30625
	 Earth_correct_params_S3.20c_AM3.325


### Read in orbital data

In [5]:
#Figure 3 or 7 data
if flag_data==0:
    dir=data_dir+'/rev8e3_for_simon'
    dirnames_const=['rev8a','rev8b', 'rev8c','']
elif flag_data==1:
    dir=data_dir+'/rev9_for_simon'
    dirnames_const=['rev9a','rev9b', 'rev9c','rev9e2']

Nruns_const=np.size(dirnames_const)

#########################
#read in the constant path
Ctime_const=[]
Ctime_high_const=[]
Ca_const=[]
Cob_const=[]
Comg_const=[]
CL_const=[]
Ce_const=[]
Ci_const=[]

for k in np.arange(np.size(dirnames_const)):
    
    ###Semi-major axis, e, i
    Cuk_f=open(dir+'/'+dirnames_const[k]+'/moon101.out', 'r')

    #extract the lines and strip them of any new lines etc.
    reader = csv.reader(Cuk_f, delimiter=" ", skipinitialspace=True)
    temp = list(reader)

    for i in np.arange(len(temp)):
        
        if (np.size(temp[i])>3):
            
            Ctime_const.append(temp[i][0]) #time
            Ca_const.append(temp[i][1]) #semi-major axis
            Ce_const.append(temp[i][2]) #eccentricity
            Ci_const.append(temp[i][3])
        

    ###obliquity
    Cuk_f=open(dir+'/'+dirnames_const[k]+'/pole.out', 'r')

    #extract the lines and strip them of any new lines etc.
    reader = csv.reader(Cuk_f, delimiter=" ", skipinitialspace=True)
    temp = list(reader)

    for i in np.arange(len(temp)):
        if np.size(temp[i])>4:
            Cob_const.append(temp[i][1]) #obliquity
            Comg_const.append(temp[i][3]) #rotation rate
            


#convert to useful type and unit
Ctime_const=np.asarray(Ctime_const, dtype=np.float64)
Ca_const=np.asarray(Ca_const, dtype=np.float64)
Cob_const=np.asarray(Cob_const, dtype=np.float64)
Ce_const=np.asarray(Ce_const, dtype=np.float64)
Ci_const=np.asarray(Ci_const, dtype=np.float64)
Comg_const=np.asarray(Comg_const, dtype=np.float64)/const.year
CL_const=Comg_const*C_Earth*MEarth*(REarth**2)

#remove any repeat values
temp=(Ctime_const[1:]-Ctime_const[:-1])
ind=np.where(temp==0.0)[0]
Ctime_const=np.delete(Ctime_const,ind+1)
Ca_const=np.delete(Ca_const,ind+1)
Cob_const=np.delete(Cob_const,ind+1)
Ce_const=np.delete(Ce_const,ind+1)
Ci_const=np.delete(Ci_const,ind+1)
Comg_const=np.delete(Comg_const,ind+1)
CL_const=np.delete(CL_const,ind+1)

#change names to be consistent with rest of program
Ctime=Ctime_const
Ca=Ca_const
Cob=Cob_const
Ce=Ce_const
Ci=Ci_const

Comg=Comg_const
CL=CL_const

    

print('done')

done


### Read in the precomputed strain data

In [6]:
#if needed read in the max integrated deformation
print('start')

#define the output files
if flag_data==0:
    data_output_file_int=data_dir+'/Cuk21_surf_change_max_integrated_CukFig3.bin'
elif flag_data==1:
    data_output_file_int=data_dir+'/Cuk21_surf_change_max_integrated_CukFig7.bin'

dataf_int = open(data_output_file_int, "rb")

#now read in the rest of the file as one massive array
ndata_per_step=6
data = np.fromfile(dataf_int, dtype=np.float64, count=-1)
dataf_int.close()

#work out how many time steps we have
temp=np.size(data)*1.0/(1.0*ndata_per_step)
print(temp)
if abs(temp-int(temp))<(1E-12):
    Ntstep=int(temp)
else:
    print("ERROR IN READ",'\n',"Not complete number of steps or incorrect number of print params",'\n',"EXITING")

#reshape array so that each row is a timestep
data=data.reshape(Ntstep,ndata_per_step)

steps_max=data[:,0].astype(int)
time_max=data[:,1]
ddl_lon_dt_max=data[:,2]
ddl_lon_dt_min=data[:,3]
ddl_lat_dt_min=data[:,4]
ddl_lat_dt_max=data[:,5]

print('end')



start
136382.0
end


### Interpolate the orbital data to create the time snapshots

In [7]:
#Interpolate to find the time, AM and a to plot

time=np.arange(0.0,np.amax(Ctime),tstep_plot*1E6)
temp=np.where(time<np.amin(Ctime))[0]
if np.size(temp)!=0:
    time=time[(temp[-1]+1):]

Nt_plot=np.size(time) #number of time points to plot
print(Nt_plot)

dLdt=gradient2(Ctime,CL)

#create 1D interpolation hulls for L and a
fCL = interpolate.interp1d(Ctime, CL, kind='linear')
fCa = interpolate.interp1d(Ctime, Ca, kind='linear')
fdLdt = interpolate.interp1d(Ctime, dLdt, kind='linear')

#interpolate to find the L and a to plot
CL_plot=fCL(time)
Ca_plot=fCa(time)
dLdt_plot=fdLdt(time)

print('done')

1001
done


### Calculate the corresponding change in length

In [8]:
#now run through all time steps and calculate the change in length
print('begin')

checkpoints=np.linspace(1,101,101)

#run through and extract lengths and areas at each time point
dl_lat=np.zeros((Nt_plot,Nmu))
dl_lon=np.zeros((Nt_plot,Nmu))
dA=np.zeros((Nt_plot,Nmu))

ddl_lat_dL=np.zeros((Nt_plot,Nmu))
ddl_lon_dL=np.zeros((Nt_plot,Nmu))
ddA_dL=np.zeros((Nt_plot,Nmu))

ddl_lat_dt=np.zeros((Nt_plot,Nmu))
ddl_lon_dt=np.zeros((Nt_plot,Nmu))
ddA_dt=np.zeros((Nt_plot,Nmu))

rsurf=np.zeros((Nt_plot,Nmu))

# dLdt=gradient2(time,CL_plot)

count=-1
for i in np.arange(Nt_plot):
    count+=1
    #print(i, np.size(steps),count)
    if (i*1.0/Nt_plot*100)>checkpoints[0]:
        print(i*1.0/Nt_plot*100, '%')
        checkpoints=checkpoints[1:]
    
    temp_data=Hdatabase.interp_database(CL_plot[i],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1],\
                                       [0,0,[0],0,0,0,0,[0],[0],[0],[0],[0],[0],[0],[0],[0],[0],[0]],flag_extrap=1)

    temp=np.asarray(temp_data[16])

    dl_lat[count,:]=temp[:,0]
    dl_lon[count,:]=temp[:,1]
    dA[count,:]=temp[:,2]
    
    ddl_lat_dL[count,:]=temp[:,3]
    ddl_lon_dL[count,:]=temp[:,4]
    ddA_dL[count,:]=temp[:,5]
    
    ddl_lat_dt[count,:]=temp[:,3]*dLdt_plot[count]
    ddl_lon_dt[count,:]=temp[:,4]*dLdt_plot[count]
    ddA_dt[count,:]=temp[:,5]*dLdt_plot[count]
    
    rsurf[i,:]=np.asarray(temp_data[17])

    
print('end')

begin
1.098901098901099 %
2.097902097902098 %
3.096903096903097 %
4.095904095904096 %
5.094905094905095 %
6.093906093906094 %
7.092907092907093 %
8.091908091908092 %
9.090909090909092 %
10.08991008991009 %
11.08891108891109 %
12.087912087912088 %
13.086913086913087 %
14.085914085914087 %
15.084915084915085 %
16.083916083916083 %
17.08291708291708 %
18.081918081918083 %
19.08091908091908 %
20.07992007992008 %
21.078921078921077 %
22.07792207792208 %
23.076923076923077 %
24.07592407592408 %
25.074925074925076 %
26.073926073926074 %
27.072927072927072 %
28.07192807192807 %
29.07092907092907 %
30.069930069930066 %
31.06893106893107 %
32.06793206793207 %
33.06693306693307 %
34.065934065934066 %
35.064935064935064 %
36.06393606393606 %
37.06293706293706 %
38.06193806193806 %
39.06093906093906 %
40.05994005994006 %
41.05894105894106 %
42.05794205794206 %
43.05694305694306 %
44.05594405594406 %
45.05494505494506 %
46.053946053946056 %
47.052947052947054 %
48.05194805194805 %
49.05094905094905 

### Run through and plot all the slides

In [9]:

print('begin')

#################################################
#version with a plot of AM and semi-major axis and latitudinal deformation
#now in a square arangement
#################################################

boxlim=18
z_plot=np.linspace(-boxlim/2, boxlim/2, num=Ncont)
x_plot=np.linspace(-boxlim/2, boxlim/2, num=Ncont)

XXX, ZZZ = np.meshgrid(x_plot, z_plot)

#do the first one twice to make sure the graphics are working
temp=np.append(0,np.arange(np.size(time))[0:])
for k in temp[0:]:
    print(k+1, str(time[k]/1E6)+' Myr', str((k+1)/Nt_plot*100)+'%')
    #loop over all points on the surface
    Nrow_data=Ncont
    x_data=np.zeros(Hdatabase.parr[0].Nmu*Nrow_data*2)
    z_data=np.zeros(Hdatabase.parr[0].Nmu*Nrow_data*2)
    ddl_lon_dt_data=np.zeros(Hdatabase.parr[0].Nmu*Nrow_data*2)
    

    for i in np.arange(Hdatabase.parr[0].Nmu):
        x=rsurf[k,i]*np.sin(np.arccos(Hdatabase.parr[0].mu[i]))
        z=rsurf[k,i]*Hdatabase.parr[0].mu[i]
        x_data[2*i*Nrow_data:(i+1)*2*Nrow_data]=np.append(np.linspace(-x, x, Nrow_data),np.linspace(-x, x, Nrow_data))
        z_data[2*i*Nrow_data:(i+1)*2*Nrow_data]=np.append(np.ones(Nrow_data)*z,-np.ones(Nrow_data)*z)
        ddl_lon_dt_data[2*i*Nrow_data:(i+1)*2*Nrow_data]=np.ones(2*Nrow_data)*ddl_lon_dt[k,i]
        
    DLON=griddata(((x_data.flatten()/1E6,z_data.flatten()/1E6)),ddl_lon_dt_data.flatten()*np.pi/180.0,(XXX,ZZZ),fill_value=np.nan)#, method='linear')
    
    #initialise the figure
    fig = plt.figure(figsize=(7.7,5.5))
    gs0 = gridspec.GridSpec(1, 2, width_ratios=[0.9,1])
    
    gs00 = gridspec.GridSpecFromSubplotSpec(3, 2,
                                            width_ratios=[1,0.05],
                                            height_ratios=[1,1,1.65],
                                    subplot_spec=gs0[0])
    gs01 = gridspec.GridSpecFromSubplotSpec(3, 2,
                                            width_ratios=[1,0.05],
                                            height_ratios=[0.5,2,0.5],
                                    subplot_spec=gs0[1])

    
    ax=[[]]
    ax[0].append(plt.subplot(gs00[0]))
    ax[0].append(plt.subplot(gs00[2], sharex=ax[0][0]))
    ax[0].append(plt.subplot(gs01[2]))
    ax[0].append(plt.subplot(gs00[4], sharex=ax[0][0]))
#         ax[0].append(plt.subplot(gs02[0]))

    ax_col=[[]]
    ax_col[0].append(plt.subplot(gs01[3]))
    
    font = {
    'family' : 'Helvetica',
            'weight' : 'normal',
            'size'   : 8}
    mpl.rc('font', **font)
    
    col=cmaps.parula([0.15,0.85])
    
    ind=np.where(Ctime<=time[k])[0]
    print(ind[-1])
    ax[0][0].plot(Ctime[ind]/1E6, Ca[ind], 'k-', linewidth=1.5)
    ax[0][1].plot(Ctime[ind]/1E6, CL[ind]/LEM, 'k-', linewidth=1.5)
    
    if k==Nt_plot-1:
        ind_max=-1
        ax[0][3].plot([0,Ctime[ind_max]/1E6],[60E-3,60E-3],':', color=col[1],linewidth=1.0, label='Average subduction') #avererage subduction
        ax[0][3].plot([0,Ctime[ind_max]/1E6],[50E-3,50E-3],':', color=col[0],linewidth=1.0, label='Average ridge') #average mid-ocean ridge
        ax[0][3].plot([0,Ctime[ind_max]/1E6],[15E-3,15E-3],'--', color=col[1],linewidth=1.0, label='Slow subduction') #slow subduction
        ax[0][3].plot([0,Ctime[ind_max]/1E6],[8E-3,8E-3],'--', color=col[0],linewidth=1.0, label='Slow ridge') #slow mid ocean ridge

    
    ax[0][3].plot(Ctime[ind]/1E6, np.absolute(ddl_lon_dt_max[ind]), '-', color=col[0], linewidth=1.5)
    ax[0][3].plot(Ctime[ind]/1E6, np.absolute(ddl_lon_dt_min[ind]), '-', color=col[1], linewidth=1.5)
    
    ax[0][3].set_yscale('log')
    ax[0][3].set_ylim([1E-3,1E4])
    
    
    if flag_data==0:
        
        ax[0][0].set_ylim([2.0,27])

        ax[0][1].set_ylim([0.4,2.3])
        ax[0][1].set_xlim([-4,128])
        ax[0][0].set_xlim([-4,128])

#             lat_contours=np.linspace(-7,-2.5,18+1)
#             lon_contours=np.linspace(-7,-2.5,18+1)
#             Acontours=np.linspace(-2,3,25+1)

#             lat_labels=np.asarray([-7,-6,-5,-4,-3])
#             lon_labels=np.asarray([-7,-6,-5,-4,-3])
#             Alabels=np.asarray([-2,-1,0,1,2,3])
    elif flag_data==1:
        ax[0][0].set_ylim([2.0,27])

        ax[0][1].set_ylim([0.4,2.1])
        ax[0][1].set_xlim([-4,108])
        ax[0][0].set_xlim([-4,108])

#             lat_contours=np.linspace(-7,-2.5,18+1)
#             lon_contours=np.linspace(-7,-2.5,18+1)
#             Acontours=np.linspace(-2,3,25+1)

#             lat_labels=np.asarray([-7,-6,-5,-4,-3])
#             lon_labels=np.asarray([-7,-6,-5,-4,-3])
#             Alabels=np.asarray([-2,-1,0,1,2,3])
        
        
    
    lat_contours=np.linspace(-7,-2.5,18+1)
    lon_contours=np.linspace(-7,-2.5,18+1)
    Acontours=np.linspace(-2,2.5,18+1)

    lat_labels=np.asarray([-7,-6,-5,-4,-3])
    lon_labels=np.asarray([-7,-6,-5,-4,-3])
    Alabels=np.asarray([-2,-1,0,1,2])
    
    contour2 = ax[0][2].contourf(XXX, ZZZ, np.log10(np.absolute(DLON)), levels=lat_contours, colors=cmaps.parula_r(np.linspace(1, 0, np.size(lat_contours)-1)))
    contour20 = ax[0][2].contour(XXX, ZZZ, DLON, linewidths=0.5, levels=[0.0], colors=['k'], linestyles='solid')
       
        
    for c in contour2.collections:
        c.set_rasterized(True)
        
    #plot the intial outline
    ax[0][2].plot(rsurf[0,:]*np.sin(np.arccos(Hdatabase.parr[0].mu))/1E6,rsurf[0,:]*Hdatabase.parr[0].mu/1E6, '--', color='k', linewidth=1.5)
    ax[0][2].plot(-rsurf[0,:]*np.sin(np.arccos(Hdatabase.parr[0].mu))/1E6,rsurf[0,:]*Hdatabase.parr[0].mu/1E6, '--', color='k', linewidth=1.5)
    ax[0][2].plot(rsurf[0,:]*np.sin(np.arccos(Hdatabase.parr[0].mu))/1E6,-rsurf[0,:]*Hdatabase.parr[0].mu/1E6, '--', color='k', linewidth=1.5)
    ax[0][2].plot(-rsurf[0,:]*np.sin(np.arccos(Hdatabase.parr[0].mu))/1E6,-rsurf[0,:]*Hdatabase.parr[0].mu/1E6, '--', color='k', linewidth=1.5)

        
    cbar2=plt.colorbar(contour2, cax=ax_col[0][0], orientation='vertical',ticks=lon_labels)
    
    cbar2.set_label(r'Rate [$\log_{10} (|$m deg$^{-1}$ yr$^{-1}$$|)$]', labelpad=-40)
    
    ax[0][2].set_aspect('equal', adjustable='box')
    
    
    #label axis
    ax[0][0].text(0.04, 0.93, 'A: '+"{:.2f}".format(round(time[k]/1E6,2))+' Myrs', horizontalalignment='left',verticalalignment='top', fontsize=10,transform=ax[0][0].transAxes, color='k')
    ax[0][1].text(0.04, 0.07, 'B', horizontalalignment='left',verticalalignment='bottom', fontsize=10,transform=ax[0][1].transAxes, color='k')
    ax[0][2].text(0.04, 0.96, 'D: Longitudinal', horizontalalignment='left',verticalalignment='top', fontsize=10,transform=ax[0][2].transAxes, color='k')
    ax[0][3].text(0.04, 0.04, 'C: Longitudinal', horizontalalignment='left',verticalalignment='bottom', fontsize=10,transform=ax[0][3].transAxes, color='k')
    

    ax[0][2].set_xlabel('[10$^6$ m]')
    
    ax[0][2].set_ylabel('[10$^6$ m]')
    
    ax[0][0].set_ylabel(r"$a_{\rm Moon}$ [$R_{\rm Earth}$]")
    ax[0][1].set_ylabel(r"AM Earth [$L_{\rm EM}$]")
    ax[0][3].set_ylabel(r"Int. deform. rate [m yr$^{-1}$]")
    ax[0][3].set_xlabel(r"Time [Myrs]")
    
    plt.setp( ax[0][0].get_xticklabels(), visible=False)
    plt.setp( ax[0][1].get_xticklabels(), visible=False)

    for i in np.arange(4):
        ax[0][i].tick_params(direction="in", top=True, right=True)


    ax_col[0][0].tick_params(direction="in")
    
    if k==Nt_plot-1:
        ax[0][3].legend(frameon=False, handlelength=1.9, loc='upper right')

    fig.tight_layout()
    if flag_data==0:
        plt.savefig(output_dir+'/MovieS3_slide_'+str(k+1).zfill(5)+'.png', dpi=600, format='png')
    elif flag_data==1:
        plt.savefig(output_dir+'/MovieS2_slide_'+str(k+1).zfill(5)+'.png', dpi=600, format='png')

    plt.close(fig)

print('done')


begin
1 0.0 Myr 0.0999000999000999%
0


/var/folders/mm/ct8d2r3j48bcp_kyghqh_n7r0000gq/T/ipykernel_3885/126194536.py:130: MatplotlibDeprecationWarning: The collections attribute was deprecated in Matplotlib 3.8 and will be removed two minor releases later.
  for c in contour2.collections:


1 0.0 Myr 0.0999000999000999%
0
2 0.1 Myr 0.1998001998001998%
20403
3 0.2 Myr 0.2997002997002997%
21477
done
